# Task 3 - Weather Scraping with Fallback

The goal is to scrape weather data from Timeanddate first.
If the first website fails, the code should handle the error and try a second weather website.
The result will be saved locally in a text file.


In [ ]:
import re
import requests
from bs4 import BeautifulSoup

In [ ]:


timeanddate_url = "https://www.timeanddate.com/weather/finland/helsinki"
wunderground_url = "https://www.wunderground.com/weather/fi/helsinki"

weather_result = ""

try:
    # First attempt: Timeanddate
    response = requests.get(timeanddate_url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    weather_result = (
        "Source: Timeanddate\n"
        "Location: Helsinki, Finland\n"
        "Weather page retrieved successfully."
    )

except requests.RequestException as error:
    print("Timeanddate failed:", error)
    print("Trying Weather Underground...")

    # Fallback: Weather Underground
    response = requests.get(wunderground_url, timeout=10)
    response.raise_for_status()

    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    page_text = soup.get_text(" ", strip=True)

    temperature_match = re.search(r"(\d+)\s*°\s*F", page_text)
    feels_like_match = re.search(r"like\s+(\d+)°", page_text)
    condition_match = re.search(
        r"like\s+\d+°\s+([A-Za-z ]+?)\s+[NSEW]{1,3}\s+\d+",
        page_text
    )

    temperature_f = temperature_match.group(1) if temperature_match else None
    feels_like_f = feels_like_match.group(1) if feels_like_match else None
    condition = condition_match.group(1).strip() if condition_match else "Not found"

    # Convert Fahrenheit to Celsius
    temperature_c = (
        round((int(temperature_f) - 32) * 5 / 9, 1)
        if temperature_f else "Not found"
    )

    feels_like_c = (
        round((int(feels_like_f) - 32) * 5 / 9, 1)
        if feels_like_f else "Not found"
    )

    weather_result = (
        "Source: Weather Underground\n"
        "Location: Helsinki, Finland\n"
        f"Temperature: {temperature_c} °C\n"
        f"Feels like: {feels_like_c} °C\n"
        f"Condition: {condition}"
    )

print(weather_result)

Timeanddate failed: 403 Client Error: Forbidden for url: https://www.timeanddate.com/weather/finland/helsinki
Trying Weather Underground...
Source: Weather Underground
Location: Helsinki, Finland
Temperature: 6.1 °C
Feels like: 6.1 °C
Condition: Fair


In [3]:
# Save Task 3 weather result to a local text file

output_path = "../output/task3_weather.txt"

with open(output_path, "w", encoding="utf-8") as file:
    file.write(weather_result)

print("Task 3 weather data saved successfully.")

Task 3 weather data saved successfully.
